In [1]:
# %pip install -q langchain langgraph langchain-groq pydantic python-dotenv
import os
from typing import Any





In [2]:
from langchain_groq import ChatGroq
from langchain.agents.middleware import before_agent, AgentState
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain.tools import tool



## 1. Setup & Authentication

In [3]:

from dotenv import load_dotenv
load_dotenv()

if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError("Set GROQ_API_KEY in your .env file")


MAIN_MODEL = "groq:openai/gpt-oss-20b"
GUARDRAIL_MODEL = "allam-2-7b" 



In [4]:
# 2. Define the Tools
@tool
def get_weather(city: str) -> str:
    """Return the (fake) current weather for a city."""
    return f"It's sunny and 27°C in {city}."

@tool("get_estate", description="Get suitable estates in the location specified.")
def get_estate(location: str) -> str:
    return f"There is a beautiful 3-bedroom villa available in {location}."

@tool("get_price", description="Get the price of an estate based on the number of bedrooms.")
def get_price(bedrooms: int) -> str:
    base_price = 850
    return f"The estimated price for a {bedrooms} bedroom apartment is ${base_price * bedrooms} per month."



In [5]:
# 3. Define the Input Guardrail Middleware
@before_agent(can_jump_to=["end"])
def real_estate_guardrail(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Guardrail: Block any requests that are not related to real estate."""
    if not state.get("messages"):
        return None

    # Grab the latest user message
    last_message = state["messages"][-1]
    message_type = last_message.get("role") if isinstance(last_message, dict) else last_message.type
    
    if message_type != "human":
        return None

    content = last_message.get("content") if isinstance(last_message, dict) else last_message.content

    # Instantiate the lightweight, fast model for classification
    classifier_llm = ChatGroq(model=GUARDRAIL_MODEL, temperature=0)
    
    guardrail_prompt = f"""You are a strict security guardrail for a real estate AI.
    Determine if the user's input is related to real estate, housing, properties, renting, or buying.
    
    User Input: "{content}"
    
    Respond with EXACTLY ONE WORD: "YES" if it is related to real estate, or "NO" if it is off-topic."""
    
    # Run the classification
    classification = classifier_llm.invoke(guardrail_prompt).content.strip().upper()
    print(f"[GUARDRAIL CHECK] Evaluated: '{content}' -> {classification}")

    # Short-circuit the agent if off-topic
    if "NO" in classification:
        return {
            "messages": [{
                "role": "assistant",
                "content": "I am a specialized real estate assistant. I can only help you with questions related to buying, renting, or evaluating properties."
            }],
            "jump_to": "end"
        }

    # Return None to allow the main agent to proceed normally
    return None



In [6]:
# 4. System Prompt for the Main Agent
SYSTEM_PROMPT = """You are a real estate support agent.

ROLE: Recommend estates based on the customer price range, needs, and likings. Answer ONLY questions related to real estate.
RULES:
- Do not recommend anything you don't know about. Use your tools to look up information.
- Do not discuss topics unrelated to real estate, even if asked directly.
FORMAT: List all the real estates that match the user's needs.
EXAMPLE:
Customer: "I want to rent 3 bedroom appartment"
Assistant: "Great! If your Price range is $850 try this apartment on the west coast, if you can afford more checkout this apartment near the university."
"""



In [7]:
# 5. Build the Agent
real_estate_agent = create_agent(
    model=MAIN_MODEL,
    tools=[get_weather, get_estate, get_price],
    system_prompt=SYSTEM_PROMPT,
    middleware=[real_estate_guardrail], 
)



## 6. Run Tests

In [8]:

if __name__ == "__main__":
    print(f"Ready.\nMain Agent: {MAIN_MODEL}\nGuardrail: {GUARDRAIL_MODEL}")
    print("-" * 50)
    
    test_inputs = [
        "Whats the price of a 3 bedroom appartment in chicago?",
        "Can you help me pick a birthday gift for my mom who loves cooking?", 
        "I am looking for a 2-bedroom house under $300,000 with a large backyard."
    ]

    for user_input in test_inputs:
        print(f"\nUser: {user_input}")
        
        result = real_estate_agent.invoke({
            "messages": [{"role": "user", "content": user_input}]
        })
        
        # Safely extract the final message content
        final_message = result["messages"][-1]
        final_text = final_message["content"] if isinstance(final_message, dict) else final_message.content
        
        print(f"Agent Response: {final_text}")

Ready.
Main Agent: groq:openai/gpt-oss-20b
Guardrail: allam-2-7b
--------------------------------------------------

User: Whats the price of a 3 bedroom appartment in chicago?
[GUARDRAIL CHECK] Evaluated: 'Whats the price of a 3 bedroom appartment in chicago?' -> YES
Agent Response: The estimated monthly rent for a 3‑bedroom apartment in Chicago is **$2,550**.

User: Can you help me pick a birthday gift for my mom who loves cooking?
[GUARDRAIL CHECK] Evaluated: 'Can you help me pick a birthday gift for my mom who loves cooking?' -> NO
Agent Response: I am a specialized real estate assistant. I can only help you with questions related to buying, renting, or evaluating properties.

User: I am looking for a 2-bedroom house under $300,000 with a large backyard.
[GUARDRAIL CHECK] Evaluated: 'I am looking for a 2-bedroom house under $300,000 with a large backyard.' -> YES
Agent Response: Sure! Could you let me know which city or region you’re looking to buy in? That way I can find 2‑bedroom